# runtime.node_process

> TODO fill in description

In [ ]:
#| default_exp runtime.node_process

In [ ]:
#| hide
from nbdev.showdoc import *; 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#|export
import asyncio
from typing import Any, Callable, Dict, List, Union, Tuple

import fbdev
from fbdev.comp.port import PortType, PortSpec, PortSpecCollection, PortID, PortCollection
from fbdev.graph.net import BaseNode, NodePort, Address, NodePortCollection
from fbdev.runtime import BaseRuntime
from fbdev.comp.port import Port
from fbdev._utils.states import StateCollection
from fbdev._utils.events import EventCollection
from fbdev._utils.attr_container import AttrContainer
from fbdev.graph.packet_registry import BasePacket, TrackedPacket, PacketRegistry

In [ ]:
#|hide
from fbdev.complib import ExecComponent
from fbdev.complib.func_component_factory import func_component

In [ ]:
#|hide
show_doc(fbdev.runtime.node_process.NodePort)

---

### NodePort

>      NodePort (_port:Port, _parent_node:BaseNode)

*Ports in a Node will be converted to NodePorts. This is to facilitate addressing. It's mostly cosmetics.*

In [ ]:
#|exporti
class NodeProcessPort(NodePort):
    """NodePorts are a wrapper around NodePorts that allows for calling put and put_value without awaiting."""
    _address_delimiter = ':'
        
    def _put(self, packet:BasePacket) -> asyncio.Task:
        return asyncio.create_task(self._port._put(packet))
    
    def _put_from_external(self, packet:BasePacket):
        return asyncio.create_task(super()._put_from_external(packet))
    
    def _put_value_from_external(self, val:Any):
        return asyncio.create_task(super()._put_value_from_external(val))

In [ ]:
#|hide
show_doc(fbdev.runtime.node_process.NodeProcessPortCollection)

---

### NodeProcessPortCollection

>      NodeProcessPortCollection
>                                 (_port_collection:fbdev.comp.port.PortCollecti
>                                 on, _parent_node:fbdev.graph.net.BaseNode)

*Initialize self.  See help(type(self)) for accurate signature.*

In [ ]:
#|exporti
class NodeProcessPortCollection(NodePortCollection):
    def __init__(self, *, _port_collection:PortCollection, _parent_node:BaseNode):
        self._port_spec_collection: PortSpecCollection = _port_collection._port_spec_collection
        self._ports: Dict[str, NodeProcessPort] = {}
        for port_type in PortType:
            setattr(self, port_type.label, AttrContainer({}, obj_name=f"{PortCollection.__name__}.{port_type.label}"))
        for port in _port_collection.iter_ports():
            self._add_port(NodeProcessPort(_port=port, _parent_node=_parent_node))

In [ ]:
#|hide
show_doc(fbdev.runtime.node_process.NodeProcess)

---

### NodeProcess

>      NodeProcess (node:fbdev.graph.net.BaseNode,
>                   stop_port:Tuple[fbdev.comp.port.PortType,str]=None)

*Helper class that provides a standard way to create an ABC using
inheritance.*

In [ ]:
#|export
class NodeProcess(BaseRuntime):
    def __init__(self, node:BaseNode, stop_port:PortID=None):
        super().__init__()
        self._node:BaseNode = node
        self._stop_port = stop_port
        self._stop_listener_task = None
        
        self._ports = NodeProcessPortCollection(
            _port_collection=self._node.ports,
            _parent_node=self._node
        )
        
        if self._stop_port is not None and self._stop_port not in self._node.ports:
            raise ValueError(f"Port {self._stop_port} does not exist in node.")

    @property
    def ports(self) -> NodeProcessPortCollection: return self._ports
    
    async def gather_outputs(self, *ports:List[Union[Union[str,NodeProcessPort], Tuple[Union[str,NodeProcessPort], int]]], return_packets:bool=False) -> List[BasePacket]:
        _ports = []
        for element in ports:
            if isinstance(element, tuple):
                if len(element) != 2 or not isinstance(element[1], int):
                    raise ValueError(f"Tuple {element} must have exactly 2 elements, the second of which must be an integer.")
                port, count = element
                _ports.extend([port]*count)
            else:
                _ports.append(element)
        ports = [self.ports[PortType.OUTPUT, port] if isinstance(port, str) else port for port in _ports]
        for port in ports:
            if not port in self._ports.get_all():
                raise ValueError(f"User provided port {port} that was not found in {self.__class__.__name__}.")
            if not port.is_output_port:
                raise ValueError(f"User provided port {port} that is not an output port.")
        get_tasks = [asyncio.create_task(port.get()) for port in ports]
        results = await asyncio.gather(*get_tasks)
        if not return_packets: 
            consume_tasks = [asyncio.create_task(packet.consume()) for packet in results]
            results = await asyncio.gather(*consume_tasks)
        return results
        
    def start(self):
        """Note: this method cannot be run from within an event loop."""
        super().start()
        raise NotImplementedError(f"{self.__class__.__name__} does not support synchronous execution.")
    
    async def astart(self):
        await super().astart()
        
        if self._stop_port is not None:
            async def stop_listener():
                try:
                    await self._node.ports[self._stop_port].get()
                    if not self._stopped: await self.stop()
                except asyncio.CancelledError: pass
            self._stop_listener_task = asyncio.create_task(stop_listener())
            
        await self._node.task_manager.exec_coros(self._node.start())
        self._started = True
    
    async def await_stop(self):
        # Tried doing this by creating an event self._stop_event.
        # For some reason this would cause all tasks to just hang forever.
        # So instead, we just wait for the node to stop.
        # Really perplexing...
        await self._node.task_manager.exec_coros(self._node.states.stopped.wait(True), print_all_exceptions=False)
    
    async def stop(self):
        await super().stop()
        if self._stop_listener_task is not None:
            self._stop_listener_task.cancel()
            try: await self._stop_listener_task
            except asyncio.CancelledError: pass
        await self._node.task_manager.exec_coros(self._node.stop(), print_all_exceptions=False)
        self._stopped = True
        
    async def await_message(self, name:str):
        await self._node.task_manager.exec_coros(self._node.await_message(name), print_all_exceptions=False)

# Examples

In [ ]:
class FooComponent(ExecComponent):
    port_specs = PortSpecCollection(
        PortSpec(PortType.MESSAGE, 'stop'),
    )
    async def _execute(self):
        print("Stopping")
        await self.send_message('stop')
        
async with NodeProcess.from_component(FooComponent, stop_port=(PortType.MESSAGE, 'stop')) as ex:
    await ex.astart()
    await ex.await_stop()

Stopping


You can put values into ports without awaiting.

In [ ]:
@func_component(loop_execution=True)
def SquareComponent(inp):
    return inp**2

async with NodeProcess.from_component(SquareComponent) as ex:
    await ex.astart()
    
    ex.ports.input.inp.put_value(2)
    print("2^2 =", await ex.ports.output.out.get_and_consume())
    
    ex.ports.input.inp.put_value(5)
    print("5^2 =", await ex.ports.output.out.get_and_consume())
    
    ex.ports.input.inp.put_value(7)
    print("7^2 =", await ex.ports.output.out.get_and_consume())

2^2 = 4
5^2 = 25
7^2 = 49


In [ ]:
ex = NodeProcess.from_component(SquareComponent)

await ex.astart()

ex.ports.input.inp.put_value(2)
ex.ports.input.inp.put_value(5)
ex.ports.input.inp.put_value(7)

val1, val2, val3 = await ex.gather_outputs(('out', 3))

print("2^2 =", val1)
print("5^2 =", val2)
print("7^2 =", val3)

await ex.stop()

2^2 = 4
5^2 = 25
7^2 = 49


In [ ]:
@func_component(loop_execution=False)
def ExceptionComponent() -> None:
    raise Exception()

try:
    async with NodeProcess.from_component(ExceptionComponent) as ex:
        await ex.aexecute()
        await ex.await_message('executed')
except Exception as e:
    print("Exception raised")

Exception raised
